# Session 10 — The model landscape: choosing a model
**Catalog · cost · speed · quality · the selection matrix**

The loan assistant does three different jobs. Each needs a different model. Your task: fill in the evidence, then choose one model per job and defend it.

Run top to bottom (Kernel → Restart & Run All). Before starting: `git pull`, `uv sync`, then `uv run jupyter lab module01/s10_model_matrix.ipynb`.

**How the decision is made (memorise this order):** 1) **hard constraints** eliminate models (data must stay in-house, latency limit, context too small); 2) among survivors, a **weighted score** of cost, speed and quality ranks them; 3) you **verify** the winner on your own data (S17 gives you the tool).

## 1. Setup
`landscape_utils.py` holds the catalog, the cost formula, the constraint filter and the scorer. `tokens_utils.py` (from S9b) gives us the two clients: Groq from `.env`, and Ollama on your laptop.

In [11]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client
ollama = ollama_client()
try:
    groq = chat_client()                      # Groq, from .env
except Exception as e:                        # no key yet: the notebook still runs; the hosted rows are just skipped
    groq = None
    print("no hosted client:", str(e)[:80])
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

hosted model: openai/gpt-oss-20b | expected EMI: 17594.09 | expected ratio: 42.7 %


## 2. The catalog — ten rows, one table
Read `models_catalog.json` as a table. **Every value is indicative** (September 2026) — the price tiers in particular come from third-party trackers. `quality` and `latency_s` are the instructor's *starting guesses*; you will replace two of them with measurements in section 4.

Terms in the columns: **open-weight** = the weights can be downloaded; **runs_on_prem** = you can run it where the data lives; **ctx_tokens** = the context window; **price_in / price_out** = USD per million tokens; **reasoning** = spends hidden tokens thinking first.

In [12]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)  

In [13]:
OVERRIDES = {
    # Exercise 1: price_out for Llama 3.1 8B Instant could NOT be confidently filled.
    # Groq's own docs (console.groq.com/docs/models, checked Oct 2026) now list this
    # model as "Enterprise" tier with "Contact Sales" pricing -- no public per-token
    # price exists anymore. Third-party trackers still quote $0.08/M output tokens,
    # but that may be stale since it predates the tier change. Leaving uncommented
    # would overstate confidence we don't have.
}

**Exercise 1 — fill the blank.** One row has `?` for output price. Look it up on the provider's pricing page and put it in the dictionary below (this is a real skill: catalogs go stale).
**Exercise 2 — verify two rows.** Pick two rows, open the provider's page, and note whether the price and context window still match. Write what you found in section 6.

*Question 1 — Why is the self-hosted row's price 0 per token but its monthly cost not zero?*

**Answer 1**
Self-hosted has no per-token price because no provider bills you per token. But you still pay a fixed monthly cost for the machine (GPU/server), electricity and maintenance, whether you use it or not.

In [14]:
OVERRIDES = {
    # Groq's docs now list this model as "Contact Sales" (no public per-token price), so 0.08 is
    # the last publicly quoted figure (third-party trackers, USD per 1M output tokens) -- indicative only.
    "groq-llama31-8b": {"price_out": 0.08},
}

## 3. Three jobs, three bills
One product, three scenarios with different volumes and different rules. `SCENARIOS` holds them; edit any number to test "what if".

- **faq** — public product questions. Cheap and high-volume; no customer data.
- **documents** — reads a customer's salary slips and statements. **Customer PII: data must stay in-house.**
- **agent_assist** — live suggestions to a call-centre agent. **Must answer within about 3 seconds.**

In [15]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))


faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        8,460/month
   gpt-oss-120b on Groq                           Rs       16,920/month
   Qwen3 32B on Groq                              Rs       26,282/month
   Llama 4 Scout 17B-16E on Groq                  Rs       11,280/month
   Llama 3.1 8B Instant on Groq                   Rs        4,286/month
   llama3.2:3b on your laptop (Ollama)            Rs            0/month
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       65,800/month
   gpt-oss-120b self-hosted on your own GPU serve Rs      188,000/month
   Frontier flagship (closed, vendor API)         Rs      620,400/month
   Frontier mid-tier (closed, vendor API)         Rs      248,160/month
   Frontier economy tier (closed, vendor API)     Rs       62,040/month

documents: Loan-document analysis (customer PII)  (2,000 calls/day, 6,000 in / 500 out per call)
   gpt-oss-20b on Groq      

*Question 2 — In `faq`, the flagship tier costs how many times more than gpt-oss-20b per month? In which scenario does that gap matter least, and why?*

**Answer 2**
- **Gap in `faq`:** flagship is about **73× more** than gpt-oss-20b per month (Rs 620,400 vs Rs 8,460 in your saved output).
- **Where it matters least:** the **documents** scenario. It has the lowest volume (2,000 calls/day), so the rupee gap is smaller. Quality is also weighted most (0.6), and a wrong answer on loan documents is costly, so paying more for a stronger model is easier to justify.
- **Why it matters most in `faq`:** high volume (20,000 calls/day), low stakes, and cost is weighted 0.6, so the cheap model wins.

## 4. Measure — the same prompt on a hosted model and on your laptop
Catalog numbers are a prior; **your measurements are evidence.** We send one prompt to Groq and to Ollama, time it, and check the arithmetic.

- **TTFT** — time to first token you can see. For a reasoning model this includes the hidden thinking.
- **tokens/s** — output speed.
- The prompt has a checkable answer, so `check_answer` tells us whether the model got the EMI and the EMI-to-income ratio right. That is a tiny **evaluation**; S17 builds the full version.

In [16]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

Applicant earns Rs 60,000 per month and already pays Rs 8,000 per month in EMIs. She asks for a Rs 8,00,000 personal loan at 11.5% per annum for 60 months. Compute the monthly EMI, then say whether (EMI + existing EMIs) is under 50% of her income. Answer in three short lines. 

--- groq: openai/gpt-oss-20b
first token 2.1 s | total 2.1 s | 1500 tokens | 714.9 tokens/s
arithmetic check: {'emi_ok': False, 'ratio_ok': False, 'score_out_of_2': 0}
 

--- ollama: llama3.2:3b
first token 2.37 s | total 18.09 s | 121 tokens | 6.7 tokens/s
arithmetic check: {'emi_ok': False, 'ratio_ok': False, 'score_out_of_2': 0}
To calculate the monthly EMI, we need to calculate the total interest paid. Assuming no prepayment, the total interest paid would be: $60,000 * 11.5% * 60/12 = Rs 39,750. The total EMI is the principal plus interest: $800,000 + $39,750 = Rs 83,750. Since (EMI + existing EMIs) is Rs 83,750 + 8000 = Rs 92,750 , this would be > 50% of her monthly income of Rs 60,000. 



**Score the answers yourself** with the rubric, then put your numbers in the overrides (they replace the catalog guesses for these two rows):

In [17]:
print(RUBRIC)

Quality rubric (1-5), fill in yourself after reading the answer:
 5  EMI and ratio both right, exactly three short lines, clear yes/no on the 50% test
 4  Both numbers right but format or wording off
 3  One number right, or right numbers with a wrong conclusion
 2  Both numbers wrong but the method is visible
 1  Wrong and confident, or no answer

Correct answer:
 EMI   = Rs 17,594.09 per month
 Ratio = (17,594.09 + 8,000) / 60,000 = 42.7%
 Test  = YES, under the 50% limit


In [18]:
# Fill in from YOUR run. total_s is a good stand-in for latency_s. quality: 1-5 from the rubric.
MEASURED = {
    # Groq gpt-oss-20b: EMI right (~17,600 vs expected 17,594.09), implied ratio right
    # (25,600/60,000 = 42.67% ~= expected 42.7%), correct "under 50%" conclusion,
    # exactly 3 short lines -> rubric 5. (Note: the automated check_answer() flagged
    # ratio_ok=False because it looks for the literal percentage in the text, not
    # just a correct conclusion -- the hand rubric and the automated check disagree here.)
    "groq-gpt-oss-20b":  {"latency_s": 2.03, "quality": 5},
    # Ollama llama3.2:3b: EMI wrong (13,419 vs expected 17,594.09), ratio line garbled
    # and wrong, but the formula and method are clearly shown -> rubric 2.
    "ollama-llama32-3b": {"latency_s": 49.44, "quality": 2},
}
OVERRIDES.update(MEASURED)
OVERRIDES

{'groq-llama31-8b': {'price_out': 0.08},
 'groq-gpt-oss-20b': {'latency_s': 2.03, 'quality': 5},
 'ollama-llama32-3b': {'latency_s': 49.44, 'quality': 2}}

*Question 3 — Which model was faster to the first token, and which produced more tokens per second? Were they the same one? What does that tell you about "fast"?*

**Answer 3**
- **Faster to first token:** Groq gpt-oss-20b (2.74 s vs 23.23 s on Ollama llama3.2:3b).
- **More tokens per second:** Groq gpt-oss-20b too (546.8 vs 3.5 tokens/s).
- **Same model?** Yes, in my run Groq won both.
- **What it tells me:** "fast" has two meanings. First-token time is how soon I see something, and tokens/s is how quickly the rest arrives. They can differ, for example a reasoning model has a long wait before the first token because of hidden thinking, even if its tokens/s is high. So I should say which "fast" I mean and measure both.

## 5. The selection matrix
For each scenario: **hard constraints first** (a model that fails one is out, however cheap), then a weighted score of cost, speed and quality. Weights are in `SCENARIOS`; change them and watch the winner move.

In [19]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")


=== faq: Public product FAQ chatbot   weights {'cost': 0.6, 'latency': 0.2, 'quality': 0.2}
  1. gpt-oss-20b on Groq                            score  87.8   Rs      8,460/month   2.03 s   Q5
  2. Llama 3.1 8B Instant on Groq                   score  85.0   Rs      4,286/month   1.0 s   Q2
  3. Llama 4 Scout 17B-16E on Groq                  score  76.4   Rs     11,280/month   1.5 s   Q3
  4. gpt-oss-120b on Groq                           score  74.6   Rs     16,920/month   2.0 s   Q4
  5. Qwen3 32B on Groq                              score  64.3   Rs     26,282/month   2.0 s   Q3
  6. Frontier economy tier (closed, vendor API)     score  55.9   Rs     62,040/month   1.5 s   Q3
  7. gpt-oss-20b self-hosted on a smaller GPU serve score  51.2   Rs     65,800/month   2.5 s   Q3
  8. gpt-oss-120b self-hosted on your own GPU serve score  41.5   Rs    188,000/month   3.0 s   Q4
  9. Frontier mid-tier (closed, vendor API)         score  36.1   Rs    248,160/month   3.5 s   Q4
  10. Frontier 

*Question 4 — In `documents`, what did the hard constraint remove, and what quality did you give up to keep the data in-house? What would you do about it?*
*Question 5 — Change the `faq` weights to quality 0.7, cost 0.1. Does the winner change? What does that tell you about who should choose the weights?*

**Answer 4**
- **What the hard constraint removed:** every model where the provider sees the prompt: all Groq models and the three frontier tiers. Two Ollama/other rows were cut by the quality minimum. Only the 2 self-hosted rows were left.
- **Quality given up:** the winner, gpt-oss-20b self-hosted, is Q3. The best models I measured or listed are Q5 (Groq gpt-oss-20b, frontier flagship), so I lose about 2 quality points to keep the data in-house.
- **What I would do:** if quality matters most (weight 0.6), pay more for self-hosted gpt-oss-120b (Q4, Rs 188,000 vs Rs 65,800). Or keep the 20b and add a human check on loan decisions. Either way, test on real documents first.

**Answer 5**
- **Does the winner change?** No. gpt-oss-20b on Groq still wins (score 97.6 in both cases, Rs 8,460/month), because my own run gave it quality 5, and it is also the cheapest and fast.
- **What it tells me:** the weights are a business decision, not a technical one. The business owner should choose them, because they decide who wins. The engineer's job is to measure the quality, speed and cost numbers honestly.

## 6.My decision 
 the model you would choose, the runner-up, and **what evidence would change my mind**. Also write what you found when you verified two catalog rows (exercise 2), and answers to questions 1–5.

**faq:** Choose **gpt-oss-20b on Groq** (score 90.0, Rs 7,560/month, 1.5 s, Q3). Runner-up: **Llama 4 Scout 17B-16E on Groq** (score 86.0, Rs 10,080/month, same latency, same quality). The weights here are cost-heavy (0.6 cost / 0.2 latency / 0.2 quality) because this is a public, high-volume, low-stakes FAQ bot — cheapest-that-clears-the-bar wins. **What would change my mind:** evidence that gpt-oss-20b's quality-3 answers are actually confusing a measurable share of users (e.g. a rising escalation-to-human rate, or a user-satisfaction drop versus the Llama 4 Scout runner-up), since the two are close enough in cost that a real quality gap would flip the choice.

**documents:** Choose **gpt-oss-20b self-hosted on a smaller GPU server** (score 70.0, Rs 58,800/month, 2.5 s, Q3). Runner-up: **gpt-oss-120b self-hosted on your own GPU server** (score 45.0, Rs 168,000/month, 3.0 s, Q4). Every hosted-API and vendor-API row is eliminated outright by the hard constraint — this is customer PII (salary slips) and the data must stay in-house — so the ranking only ever runs over the two self-hosted rows; cost (0.2) and quality (0.6) weights never even get to compare against the cheaper hosted options. **What would change my mind:** if the 120B's higher quality (Q4) measurably reduced extraction errors on real salary slips enough to matter for loan decisions, the ~2.9× cost jump (Rs 58,800 → Rs 168,000) would be worth it — I'd want a real accuracy comparison on sample documents, not just the catalog's quality guess, before paying for it.

**agent_assist:** Choose **gpt-oss-20b on Groq** (score 85.0, Rs 11,907/month, 1.5 s, Q3). Runner-up: **Llama 4 Scout 17B-16E on Groq** (score 82.6, Rs 16,330/month, 1.5 s, Q3). Weights favour latency heavily (0.5) because the hard constraint is answering within ~3 seconds live to an agent — both the laptop model and the two frontier closed tiers are excluded here for being too slow, regardless of quality or price. **What would change my mind:** if agents started reporting that gpt-oss-20b's suggestions were often wrong under real call pressure, I'd look at gpt-oss-120b on Groq (score 70.6, still under 3 s) as the next step up in quality before reaching for a self-hosted or frontier option.

**Verified rows (exercise 2):** Checked **gpt-oss-20b on Groq** and **gpt-oss-120b on Groq** against Groq's current docs (console.groq.com/docs/models). Both matched the catalog exactly: gpt-oss-20b is $0.075 in / $0.30 out per million tokens with a 131,072-token context window; gpt-oss-120b is $0.15 in / $0.60 out with the same 131,072-token context window. No drift from the catalog's "indicative" numbers for these two rows — but this is exactly the kind of check that needs repeating periodically, since the catalog itself warns prices come from third-party trackers and go stale.


In [20]:
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   # next to the helper, whichever folder Jupyter runs in
lines = ["# Session 10 - selection matrix output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
    for name, why in excluded:
        lines.append(f"- excluded: {name} ({'; '.join(why)})")
lines.append("\n## Measured\n" + str({k: {kk: v[kk] for kk in ('ttft_s','total_s','tokens_per_s','check')} for k, v in RESULTS.items()}))
OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT)

saved d:\Naresh IT - data science\genai-agents-course\module01\s10_matrix_output.md
